# Business Questions :-
1. Which departments have the highest number of products ordered?
2. Which aisles have the highest product order frequency?
3. Which products are most frequently ordered?
4. Which department–aisle combinations have the highest number of orders?
5. Which day of the week has the highest number of orders?
6. At what hour of the day are orders most frequently placed?
7. Which products are most frequently added first to the cart?
8. Are products added first to the cart more likely to be reordered?
9. Which customers place orders most frequently (minimum days between orders)?

In [72]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Instacart Sales Analysis") \
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .getOrCreate()

print("Spark started")

df = spark.read.parquet("instamart_dataset.parquet")

Spark started


In [73]:
from pyspark.sql.functions import col

df = df.withColumn("order_dow",col("order_dow").cast("int"))

df = df.withColumn("order_hour_of_day",col("order_hour_of_day").cast("int"))

df = df.fillna({"days_since_prior_order":0})

In [74]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "day_name",
    when(col("order_dow") == 0, "Sunday")
    .when(col("order_dow") == 1, "Monday")
    .when(col("order_dow") == 2, "Tuesday")
    .when(col("order_dow") == 3, "Wednesday")
    .when(col("order_dow") == 4, "Thursday")
    .when(col("order_dow") == 5, "Friday")
    .when(col("order_dow") == 6, "Saturday")
)

In [75]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- add_to_cart_order: long (nullable = true)
 |-- reordered: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: long (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = false)
 |-- product_name: string (nullable = true)
 |-- aisle_id: long (nullable = true)
 |-- department_id: long (nullable = true)
 |-- aisle: string (nullable = true)
 |-- department: string (nullable = true)
 |-- day_name: string (nullable = true)



In [76]:
# Which departments have the highest number of products ordered?

from pyspark.sql.functions import count

df.groupBy("department") \
  .agg(count("*").alias("total_orders")) \
  .orderBy("total_orders", ascending=False) \
  .show()

+---------------+------------+
|     department|total_orders|
+---------------+------------+
|        produce|     9479291|
|     dairy eggs|     5414016|
|         snacks|     2887550|
|      beverages|     2690129|
|         frozen|     2236432|
|         pantry|     1875577|
|         bakery|     1176787|
|   canned goods|     1068058|
|           deli|     1051249|
|dry goods pasta|      866627|
|      household|      738666|
|      breakfast|      709569|
|   meat seafood|      708931|
|  personal care|      447123|
|         babies|      423802|
|  international|      269253|
|        alcohol|      153696|
|           pets|       97724|
|        missing|       69145|
|          other|       36291|
+---------------+------------+
only showing top 20 rows


In [77]:
# Which aisles have the highest product order frequency?

df.groupBy("aisle") \
  .agg(count("*").alias("total_orders")) \
  .orderBy("total_orders", ascending=False) \
  .show()

+--------------------+------------+
|               aisle|total_orders|
+--------------------+------------+
|        fresh fruits|     3642188|
|    fresh vegetables|     3418021|
|packaged vegetabl...|     1765313|
|              yogurt|     1452343|
|     packaged cheese|      979763|
|                milk|      891015|
|water seltzer spa...|      841533|
|      chips pretzels|      722470|
|     soy lactosefree|      638253|
|               bread|      584834|
|        refrigerated|      575881|
|      frozen produce|      522654|
|       ice cream ice|      498425|
|            crackers|      458838|
| energy granola bars|      456386|
|                eggs|      452134|
|          lunch meat|      395130|
|        frozen meals|      390299|
|   baby food formula|      382456|
|         fresh herbs|      377741|
+--------------------+------------+
only showing top 20 rows


In [78]:
# Which products are most frequently ordered?

df.groupBy("product_name") \
  .agg(count("*").alias("order_frequency")) \
  .orderBy("order_frequency", ascending=False) \
  .show()

+--------------------+---------------+
|        product_name|order_frequency|
+--------------------+---------------+
|              Banana|         472565|
|Bag of Organic Ba...|         379450|
|Organic Strawberries|         264683|
|Organic Baby Spinach|         241921|
|Organic Hass Avocado|         213584|
|     Organic Avocado|         176815|
|         Large Lemon|         152657|
|        Strawberries|         142951|
|               Limes|         140627|
|  Organic Whole Milk|         137905|
| Organic Raspberries|         137057|
|Organic Yellow Onion|         113426|
|      Organic Garlic|         109778|
|    Organic Zucchini|         104823|
| Organic Blueberries|         100060|
|      Cucumber Kirby|          97315|
|  Organic Fuji Apple|          89632|
|       Organic Lemon|          87746|
|Apple Honeycrisp ...|          85020|
|Organic Grape Tom...|          84255|
+--------------------+---------------+
only showing top 20 rows


In [79]:
#  Which department–aisle combinations have the highest number of orders?

df.groupBy("department", "aisle") \
  .agg(count("*").alias("total_orders")) \
  .orderBy("total_orders", ascending=False) \
  .show()

+----------+--------------------+------------+
|department|               aisle|total_orders|
+----------+--------------------+------------+
|   produce|        fresh fruits|     3642188|
|   produce|    fresh vegetables|     3418021|
|   produce|packaged vegetabl...|     1765313|
|dairy eggs|              yogurt|     1452343|
|dairy eggs|     packaged cheese|      979763|
|dairy eggs|                milk|      891015|
| beverages|water seltzer spa...|      841533|
|    snacks|      chips pretzels|      722470|
|dairy eggs|     soy lactosefree|      638253|
|    bakery|               bread|      584834|
| beverages|        refrigerated|      575881|
|    frozen|      frozen produce|      522654|
|    frozen|       ice cream ice|      498425|
|    snacks|            crackers|      458838|
|    snacks| energy granola bars|      456386|
|dairy eggs|                eggs|      452134|
|      deli|          lunch meat|      395130|
|    frozen|        frozen meals|      390299|
|    babies| 

In [80]:
# Which day of the week has the highest number of orders?

df.groupBy("day_name") \
  .agg(count("*").alias("total_orders")) \
  .orderBy("total_orders", ascending=False) \
  .show()

+---------+------------+
| day_name|total_orders|
+---------+------------+
|   Sunday|     6209666|
|   Monday|     5665856|
| Saturday|     4500304|
|  Tuesday|     4217798|
|   Friday|     4209533|
|Wednesday|     3844117|
| Thursday|     3787215|
+---------+------------+



In [81]:
# At what hour of the day are orders most frequently placed?

df.groupBy("order_hour_of_day") \
  .agg(count("*").alias("total_orders")) \
  .orderBy("total_orders", ascending=False) \
  .show()

+-----------------+------------+
|order_hour_of_day|total_orders|
+-----------------+------------+
|               10|     2764426|
|               11|     2738582|
|               14|     2691548|
|               15|     2664533|
|               13|     2663292|
|               12|     2620847|
|               16|     2537458|
|                9|     2456713|
|               17|     2089465|
|                8|     1719973|
|               18|     1637923|
|               19|     1259401|
|               20|      977038|
|                7|      891937|
|               21|      796370|
|               22|      634734|
|               23|      402620|
|                6|      290795|
|                0|      218948|
|                1|      115786|
+-----------------+------------+
only showing top 20 rows


In [82]:
# Which products are most frequently added first to the cart?

df.filter(df.add_to_cart_order == 1) \
  .groupBy("product_name") \
  .agg(count("*").alias("first_add_count")) \
  .orderBy("first_add_count", ascending=False) \
  .show()

+--------------------+---------------+
|        product_name|first_add_count|
+--------------------+---------------+
|              Banana|         110916|
|Bag of Organic Ba...|          78988|
|  Organic Whole Milk|          30927|
|Organic Strawberries|          27975|
|Organic Hass Avocado|          24116|
|Organic Baby Spinach|          23543|
|     Organic Avocado|          22398|
|        Spring Water|          16822|
|        Strawberries|          16366|
| Organic Raspberries|          14393|
|Sparkling Water G...|          13733|
| Organic Half & Half|          12676|
|         Large Lemon|          12316|
|                Soda|          11770|
|Organic Reduced F...|           9885|
|               Limes|           9719|
|         Half & Half|           9528|
|       Hass Avocados|           9500|
|Organic Reduced F...|           9338|
|         Raspberries|           8885|
+--------------------+---------------+
only showing top 20 rows


In [83]:
# Are products added first to the cart more likely to be reordered?

from pyspark.sql.functions import avg

df.groupBy("add_to_cart_order") \
  .agg(avg("reordered").alias("reorder_rate")) \
  .orderBy("add_to_cart_order") \
  .show()

+-----------------+-------------------+
|add_to_cart_order|       reorder_rate|
+-----------------+-------------------+
|                1| 0.6775329297509016|
|                2| 0.6762507496421011|
|                3| 0.6580367401997748|
|                4| 0.6369577636925858|
|                5| 0.6173831144234805|
|                6| 0.6004201120750601|
|                7| 0.5856874553126353|
|                8| 0.5732474374495332|
|                9| 0.5614735319715354|
|               10|  0.551017816966349|
|               11| 0.5410140483185638|
|               12| 0.5325829217052386|
|               13| 0.5247755707923941|
|               14| 0.5163748682407989|
|               15| 0.5091902382114232|
|               16| 0.5029074631860776|
|               17| 0.4960078621241061|
|               18| 0.4908220884013716|
|               19|0.48490486632555874|
|               20| 0.4798950800270835|
+-----------------+-------------------+
only showing top 20 rows


In [84]:
# Which customers place orders most frequently (minimum days between orders)?

from pyspark.sql.functions import avg

df.groupBy("user_id") \
  .agg(avg("days_since_prior_order").alias("avg_days_between_orders")) \
  .orderBy("avg_days_between_orders") \
  .show()

+-------+-----------------------+
|user_id|avg_days_between_orders|
+-------+-----------------------+
| 115420|                    0.0|
|  80567|                    0.0|
|  58934|                    0.0|
| 125228|                    0.0|
|  14433|                    0.0|
| 109010|                    0.0|
|  36904|                    0.0|
|  64975|                    0.0|
| 201321|                    0.0|
|  15495|                    0.0|
| 179078|                    0.0|
| 174627|                    0.0|
| 121915|                    0.0|
|  62180|                    0.0|
| 137150|                    0.0|
|  74329|                    0.0|
| 131603|                    0.0|
| 133075|                    0.0|
| 164320|                    0.0|
|  74660|                    0.0|
+-------+-----------------------+
only showing top 20 rows
